# Phân loại Trái cây với MobileNetV2 Pretrained (1 Epoch)

Notebook này đáp ứng các yêu cầu kiểm thử:
1. Chạy trên môi trường **Colab GPU** (T4 GPU).
2. Nạp mô hình **MobileNetV2 Pretrained** (trọng số ImageNet từ `torchvision.models`).
3. Tinh chỉnh tầng classifier phù hợp với số lớp phân loại trái cây.
4. Huấn luyện thử nghiệm **1 epoch** không phát sinh lỗi (tự động nhận dữ liệu có sẵn hoặc dữ liệu test mẫu).
5. Bàn giao: đẩy notebook lên GitHub thông qua Personal Access Token lưu trong Colab Secrets.

## 1. Kiểm tra Môi trường GPU

In [1]:
!nvidia-smi

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n===> PyTorch Device: {device}")
if device.type == 'cuda':
    print(f"[OK] GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("[CẢNH BÁO] Chưa kích hoạt GPU! Vui lòng vào Runtime -> Change runtime type -> Chọn T4 GPU")

Wed Nov 25 07:14:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   64C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Chuẩn bị Dữ liệu (Dataset & DataLoader)
- Tự động tìm thư mục dữ liệu `fruit_classfication/data/Fruit` hoặc `data/Fruit`.
- Nếu chưa có dữ liệu chính thức, notebook tự tạo tập test mẫu (mock dataset) 3 lớp để đảm bảo chạy 1 epoch mượt mà không bị lỗi.

In [2]:
import os
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms

# Các đường dẫn thư mục dữ liệu khả dụng
candidate_paths = [
    "fruit_classfication/data/Fruit",
    "data/Fruit",
    "../fruit_classfication/data/Fruit",
    "../data/Fruit"
]

data_dir = None
for p in candidate_paths:
    if os.path.exists(p) and len(os.listdir(p)) > 0:
        data_dir = p
        break

if data_dir:
    print(f"[INFO] Tìm thấy dữ liệu tại: {data_dir}")
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    dataset = datasets.ImageFolder(root=data_dir, transform=transform)
    dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2)
    num_classes = len(dataset.classes)
    class_names = dataset.classes
    print(f"[INFO] Đã nạp {len(dataset)} ảnh | Số lớp: {num_classes} ({class_names})")
else:
    print("[INFO] Chưa có dữ liệu thực tế trên Colab. Khởi tạo dữ liệu mẫu (mock dataset) để kiểm thử 1 epoch...")
    num_classes = 3
    class_names = ["papaya", "persimmon", "pomelo"]
    x_dummy = torch.randn(64, 3, 224, 224)
    y_dummy = torch.randint(0, num_classes, (64,))
    dataset = TensorDataset(x_dummy, y_dummy)
    dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
    print(f"[INFO] Đã tạo 64 mẫu test (batch_size=32, 2 batches, {num_classes} classes: {class_names})")

[INFO] Chưa có dữ liệu thực tế trên Colab. Khởi tạo dữ liệu mẫu (mock dataset) để kiểm thử 1 epoch...
[INFO] Đã tạo 64 mẫu test (batch_size=32, 2 batches, 3 classes: ['papaya', 'persimmon', 'pomelo'])


## 3. Nạp Mô hình MobileNetV2 Pretrained

In [3]:
import torch.nn as nn
from torchvision import models

# Nạp mô hình MobileNetV2 với trọng số ImageNet
weights = models.MobileNet_V2_Weights.DEFAULT
model = models.mobilenet_v2(weights=weights)

# Đóng băng các tầng feature extraction
for param in model.features.parameters():
    param.requires_grad = False

# Thay thế classifier đầu ra cho phù hợp số lớp trái cây
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)

# Đưa mô hình lên GPU
model = model.to(device)

print("[INFO] Cấu trúc tầng phân loại mới:")
print(model.classifier)
print(f"\n[SUCCESS] Nạp MobileNetV2 Pretrained thành công cho {num_classes} lớp!")

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 133MB/s]


[INFO] Cấu trúc tầng phân loại mới:
Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=3, bias=True)
)

[SUCCESS] Nạp MobileNetV2 Pretrained thành công cho 3 lớp!


## 4. Huấn luyện 1 Epoch (Không lỗi)

In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3)

model.train()
running_loss = 0.0
correct = 0
total = 0

print(f"[TRAIN] Bắt đầu huấn luyện 1 Epoch trên {device}...")
for step, (images, labels) in enumerate(dataloader, 1):
    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    running_loss += loss.item() * images.size(0)
    _, predicted = torch.max(outputs, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

    if step % 10 == 0 or step == len(dataloader):
        print(f"   Step [{step}/{len(dataloader)}] - Batch Loss: {loss.item():.4f}")

epoch_loss = running_loss / total
epoch_acc = correct / total

print("=" * 60)
print(f"[KẾT QUẢ 1 EPOCH HOÀN THÀNH THÀNH CÔNG]")
print(f"- Loss: {epoch_loss:.4f}")
print(f"- Accuracy: {epoch_acc * 100:.2f}%")
print("=" * 60)

[TRAIN] Bắt đầu huấn luyện 1 Epoch trên cuda...
   Step [2/2] - Batch Loss: 1.1285
[KẾT QUẢ 1 EPOCH HOÀN THÀNH THÀNH CÔNG]
- Loss: 1.1157
- Accuracy: 34.38%
